In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['subtype_for_TCGA-KIRC.tsv',
 "samples_for_TCGA-BRCA_subtype_'lobular'_tumor_'other'_subtype_tissue_'breast_lobular'_sstage_'missing'.tsv",
 'degs.txt',
 'cases_for_TCGA-LUSC.tsv',
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'cases_for_TCGA-THYM.tsv',
 'cases_for_TCGA-PAAD.tsv',
 'cases_for_TCGA-LAML.tsv',
 'cases_for_TCGA-LIHC.tsv',
 'subtype_for_TCGA-KIRP.tsv',
 'subtype_for_TCGA-CHOL.tsv',
 'cases_for_TCGA-MESO.tsv',
 'cases_for_TCGA-BLCA.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_157ec9e0-ca6e-4da1-9233-f976cf0db433_sample_type_Primary_Tumor_stage_Stage_IV_fileid_a127fb1e-aa3b-4b6d-98b5-c141ffba9ae7.tsv',
 'cases_for_TCGA-KIRP.tsv',
 'cases_for_TCGA-LGG.tsv',
 'cases_for_TCGA-ESCA.tsv',
 'cases_for_TCGA-GBM.tsv',
 'subtype_for_TCGA-ACC.tsv',
 'cases_for_TCGA-TGCT.tsv',
 'subtype_for_TCGA-MESO.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_1783cac1-253a-40af-a9ac-48dfb20e1ab8_sample_type_Primary_Tumor_stage_Stage_IV_fileid_451ecdf8-433f-4d38

### Methods

#### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

#### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

#### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

#### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

#### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

#### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(df_psi))
df_psi.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
df_psi.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [9]:
i = 3
pid = df_psi.iloc[i].pid
primary_site = df_psi.iloc[i].primary_site

print(i, pid, primary_site)

3 TCGA-LGG Brain


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [10]:
force=False
verbose=True

pid = df_psi.iloc[i].pid
primary_site = df_psi.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

3 TCGA-LGG Brain
Table opened ((516, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LGG.tsv'
df_cases 516 df_subt 2


### df_subt

In [11]:
df_subt

,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-LGG,other,glioma,other,missing,466
1,TCGA-LGG,other,other,other,missing,50


### df_cases

In [12]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Brain,Gliomas,2cf49bac-dea1-45ff-9e1d-09f5a2ac5215,"[{'primary_diagnosis': 'Oligodendroglioma, NOS...",TCGA-LGG,other,unknown,"Oligodendroglioma, NOS",G3,NaN,...,gliomas,oligodendroglioma,glioma,other,other,ok,ambiguous,1,0.001938,missing
1,Brain,Gliomas,183dd089-e932-4be2-b252-0e8572e7da4e,"[{'primary_diagnosis': 'Mixed glioma', 'tumor_...",TCGA-LGG,other,unknown,Mixed glioma,G2,NaN,...,gliomas,mixed glioma,glioma,other,other,ok,ambiguous,1,0.001938,missing
2,Brain,Gliomas,1f33aebc-edc1-42c5-baa9-e4b0a06fcb26,"[{'primary_diagnosis': 'Astrocytoma, NOS', 'tu...",TCGA-LGG,other,unknown,"Astrocytoma, NOS",G2,NaN,...,gliomas,astrocytoma,glioma,other,other,ok,ambiguous,1,0.001938,missing


In [13]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-LGG,2cf49bac-dea1-45ff-9e1d-09f5a2ac5215,other,glioma,other,unknown
1,TCGA-LGG,183dd089-e932-4be2-b252-0e8572e7da4e,other,glioma,other,unknown
2,TCGA-LGG,1f33aebc-edc1-42c5-baa9-e4b0a06fcb26,other,glioma,other,unknown
3,TCGA-LGG,cb598780-9e42-4167-b487-eec90ad4f36f,other,glioma,other,unknown
4,TCGA-LGG,dc7ec60b-5ea5-48a4-8908-55caf9214272,other,glioma,other,unknown
5,TCGA-LGG,60b30a95-3e4c-413a-9de6-16a5bca163c4,other,glioma,other,unknown
6,TCGA-LGG,b7cd3b44-ef6a-4207-a44f-14f5b3b56ed4,other,glioma,other,unknown
7,TCGA-LGG,a0b7c7c8-a6f2-41ba-9e61-9b7f8fbbbe1f,other,glioma,other,unknown
8,TCGA-LGG,c16d9f69-26f9-4ec7-b2d5-ff9f7dacfe0f,other,glioma,other,unknown
9,TCGA-LGG,e09f93e1-1649-48fa-9e2e-d13ac6d50b97,other,glioma,other,unknown


### Get all cases and their classifications

In [14]:

force=True
verbose=True

run_all = False

if run_all:

    for i in range(len(df_psi)):
        print(i, end=' ')
        pid = df_psi.iloc[i].pid
        primary_site = df_psi.iloc[i].primary_site

        print(pid, primary_site)

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)



### Search for samples and files for each case given

- input: pid, subtype_global, tumor_class, subtype_tissue, and stage

In [15]:
force=False
verbose=True

ipsi=5
pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)
df_subt

3 TCGA-BRCA Brain
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'


,pid,subtype_global,tumor_class,subtype_tissue,sstage,n
0,TCGA-BRCA,other,other,other,II,447
1,TCGA-BRCA,lobular,other,breast_lobular,missing,223
2,TCGA-BRCA,other,other,other,III,154
3,TCGA-BRCA,other,other,other,I,141
4,TCGA-BRCA,other,other,other,missing,85
5,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,missing,19
6,TCGA-BRCA,other,other,other,IV,16
7,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,missing,6
8,TCGA-BRCA,ductal,other,breast_ductal,II,2
9,TCGA-BRCA,clear_cell,other,clear_cell,II,2


In [37]:
cols

['pid', 'case_id', 'subtype_global', 'tumor_class', 'subtype_tissue', 'stage']

In [44]:
cols = ['pid', 'subtype_global', 'tumor_class', 'subtype_tissue', 'stage']
df_subt2 = df_cases[cols].drop_duplicates()
df_subt2 = df_subt2.sort_values(cols).reset_index(drop=True)
df_subt2

,pid,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,Stage X
1,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,unknown
2,TCGA-BRCA,clear_cell,other,clear_cell,Stage IIA
3,TCGA-BRCA,ductal,adenocarcinoma,breast_ductal,unknown
4,TCGA-BRCA,ductal,other,breast_ductal,Stage IIA
5,TCGA-BRCA,ductal,other,breast_ductal,Stage IIB
6,TCGA-BRCA,ductal,other,breast_ductal,unknown
7,TCGA-BRCA,lobular,other,breast_lobular,unknown
8,TCGA-BRCA,neuroendocrine,neuroendocrine_tumor,neuroendocrine,unknown
9,TCGA-BRCA,other,other,other,Stage I


In [45]:
isubt = 4
df_subt2.iloc[isubt]

pid                   TCGA-BRCA
subtype_global           ductal
tumor_class               other
subtype_tissue    breast_ductal
stage                 Stage IIA
Name: 4, dtype: object

In [ ]:
force=False
verbose=True

isubt=0
row = df_subt2.iloc[isubt]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
stage = row.stage

s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{stage}'"
print(s_case)

df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             sstage=stage, batch_size=200,
                                             force=force, verbose=verbose)

print(len(df_samples))

df_samples.head(3)


TCGA-BRCA subtype 'adenocarcinoma_generic' tumor 'adenocarcinoma' subtype_tissue 'adenocarcinoma_generic' stage 'Stage X'
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
>>> 1 cases [6e6408d6-6c48-4dfc-9f1c-28b7386e87b4] .....
Searching: 0-1 
>>> 1 ['6e6408d6-6c48-4dfc-9f1c-28b7386e87b4']
..

👉 Returned 81 / Total paginated 81
Table saved ((84, 11)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-BRCA_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'adenocarcinoma_generic'_sstage_'Stage_X'.tsv'
84


,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
0,6e6408d6-6c48-4dfc-9f1c-28b7386e87b4,2bf07767-58bd-4eb8-a0ff-434a9fdf5c8e,Primary Tumor,308f5292-62ca-4bdc-9afc-382c3be21149,3c80b72c-abaa-4057-a720-150ba006682e.svaba.som...,Structural Rearrangement,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,Stage X
1,6e6408d6-6c48-4dfc-9f1c-28b7386e87b4,2bf07767-58bd-4eb8-a0ff-434a9fdf5c8e,Primary Tumor,2bcf64b5-8d4e-4761-98f0-a122586e9455,b2868132-6476-4eec-9032-2a668d894baa.rna_seq.g...,Aligned Reads,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,Stage X
2,6e6408d6-6c48-4dfc-9f1c-28b7386e87b4,2bf07767-58bd-4eb8-a0ff-434a9fdf5c8e,Primary Tumor,a0d721fe-5411-4bd1-8052-72ba590591e1,0a7a17d5-f136-49f7-b457-2a314d3c991d.wxs.aliqu...,Masked Somatic Mutation,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,Stage X


In [48]:
df_cases.stage.unique()

array(['unknown', 'Stage IIIA', 'Stage IA', 'Stage IIB', 'Stage IIA',
       'Stage IIIB', 'Stage IIIC', 'Stage I', 'Stage IV', 'Stage X',
       'Stage II', 'Stage IB'], dtype=object)

In [49]:
np.unique(df_samples.data_type)

array(['Aggregated Somatic Mutation', 'Aligned Reads',
       'Allele-specific Copy Number Segment',
       'Annotated Somatic Mutation', 'Copy Number Segment',
       'Gene Expression Quantification', 'Gene Level Copy Number',
       'Intermediate Analysis Archive',
       'Isoform Expression Quantification', 'Masked Copy Number Segment',
       'Masked Intensities', 'Masked Somatic Mutation',
       'Methylation Beta Value', 'Pathology Report',
       'Protein Expression Quantification', 'Raw Intensities',
       'Raw Simple Somatic Mutation', 'Simple Germline Variation',
       'Slide Image', 'Splice Junction Quantification',
       'Structural Rearrangement', 'Transcript Fusion',
       'miRNA Expression Quantification'], dtype=object)

In [50]:
data_type = 'Gene Expression Quantification'
df_samples2 = df_samples[df_samples.data_type == data_type]
print(len(df_samples2))
df_samples2

1


,case_id,sample_id,sample_type,file_id,file_name,data_type,pid,subtype_global,tumor_class,subtype_tissue,stage
47,6e6408d6-6c48-4dfc-9f1c-28b7386e87b4,2bf07767-58bd-4eb8-a0ff-434a9fdf5c8e,Primary Tumor,b96a8979-77c1-41b5-a247-0342262d30f7,b2868132-6476-4eec-9032-2a668d894baa.rna_seq.a...,Gene Expression Quantification,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,Stage X


In [27]:
df_samples2.groupby(['case_id', 'sample_type', 'data_type']).size()

case_id                               sample_type    data_type                     
08740d7f-5a5e-4dfa-bd48-7fbf228a7a28  Primary Tumor  Gene Expression Quantification    1
5c59028f-b8fa-4811-8314-be3eaed5f364  Primary Tumor  Gene Expression Quantification    1
aba5f46a-e67a-4cd2-9c52-c0686968ff04  Primary Tumor  Gene Expression Quantification    1
e7a00d67-2c26-4d1f-bd17-35f659e88bc1  Primary Tumor  Gene Expression Quantification    1
dtype: int64

In [ ]:
pid

In [ ]:
df_cases, _, _ = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, 
											         force=False, verbose=True)

cols = ['subtype_global', 'tumor_class', 'subtype_tissue', 'stage']
df_cases[cols]

In [ ]:
gdc.df_cases.head(2)

In [ ]:
dfu = gdc.group_file_types(df_samples)
dfu

In [ ]:
lista = list(dfu.data_type)
lista.sort()

"; ".join(lista)

### Get Any table (for tumor and control)

In [ ]:
force=False
verbose=True

isamp=0
row = df_samples.iloc[isamp]

pid = row.pid
case_id = row.case_id
file_type = row.data_type
file_id = row.file_id
sample_type = row.sample_type
stage = row.stage


print(f"{pid}, {case_id}, {file_type}, {file_id}")

dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                 file_type=file_type, sample_type=sample_type, stage=stage, file_id=file_id, 
                                 force=force, verbose=verbose)
print(len(dft))
dft.head(6)

### Get Expression table (for tumor and control)

#### 🧪 Mental model (important)

| Step	|  Needs strand? | Output | Column |
|---------|---------|---------|---------|
| Read assignment	|  YES |  counts | stranded_first |
| Normalization	| NO | TPM / FPKM | both: unstranded |
| Biological meaning | NO | gene expression| both: unstranded |

In [ ]:
file_type = 'Gene Expression Quantification'

force=False
verbose=False

dft = pd.DataFrame()

df2 = df_samples[df_samples.data_type == file_type]

for i, row in df2.iterrows():
    pid = row.pid
    case_id = row.case_id
    file_type = row.data_type
    file_id = row.file_id
    sample_type = row.sample_type
    stage = row.stage

    print(f"{i}) {pid}, {case_id}, {file_type}, {file_id}")

    dft = gdc.get_table_given_fileID(pid=pid, case_id=case_id, 
                                     sample_type=sample_type, stage=stage, 
                                     file_type=file_type, file_id=file_id, 
                                     force=force, verbose=verbose)


dft.head(6)



In [ ]:
np.unique(df_samples.data_type)

### Get Normal and Tumor datasets given case_id and data_type

In [ ]:
df_samples.head(2)

In [ ]:
df_samples.columns

In [ ]:
data_type = 'Gene Expression Quantification'

case_id_list = np.unique(df_samples.case_id)
print(len(case_id_list))

df_normal, df_tumor = gdc.get_tumor_normal_tables(df_samples=df_samples, case_id_list=case_id_list, data_type=data_type)

len(df_normal), len(df_tumor)

In [ ]:
df_normal

In [ ]:
df_tumor

### Merge Expression Tables for DEGs' calculation

In [ ]:
dfa_normal, dfa_tumor = gdc.merge_normal_tumor_tables(pid=pid, df_normal=df_normal, df_tumor=df_tumor)

dfa_normal.shape, dfa_tumor.shape

In [ ]:
dfa_normal.head(3)

In [ ]:
dfa_tumor.head(3)

In [ ]:
i=100
dfa_tumor.iloc[i:i+30]

### Calc DEGs

In [ ]:
cdegs = CALC_DEGS(root_data=root_data)

cdegs.libs_dir, cdegs.root_data

In [ ]:
dfa_normal2 = cdegs.deduplicate_by_max_reads(dfa_normal)
len(dfa_normal2), len(dfa_normal)

In [ ]:
dfa_tumor2 = cdegs.deduplicate_by_max_reads(dfa_tumor)
len(dfa_tumor2), len(dfa_tumor)

In [ ]:
df_lfc = cdegs.run_deg_rscript(df_tumor=dfa_tumor2, df_normal=dfa_normal2,
                                method="edger",  manual_dispersion=0.1, min_total_count=10, 
                                merge_how="inner", keep_temp=False)
print(len(df_lfc))

In [ ]:
df_lfc = df_lfc.rename(columns={"log2FoldChange": "lfc", "padj": "fdr"})
df_lfc.head(3)

In [ ]:
lfc_cutoff = 1.0
fdr_cutoff = .05

df_degs = df_lfc[ (df_lfc.lfc >= lfc_cutoff) & (df_lfc.fdr < fdr_cutoff)].copy()
print(len(df_degs))

df_degs.head(3)

In [ ]:
fname = 'degs.txt'

text = "\n".join(df_degs.symbol)

write_txt(text, fname, root_data)

In [ ]:
TCGA BRCA breast cancer, subtypes 'other', stage IV